In [ ]:
!pip install transformer-lens sae-lens datasets scikit-learn torch

In [15]:
from sae_lens import SAE, HookedSAETransformer
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = HookedSAETransformer.from_pretrained_no_processing("gpt2", device=device)

release = "gpt2-small-res-jb"
sae_id = "blocks.10.hook_resid_pre"
sae = SAE.from_pretrained(release, sae_id)
sae.to(device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer


/usr/local/lib/python3.12/dist-packages/sae_lens/saes/sae.py:253: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


StandardSAE(
  (activation_fn): ReLU()
  (hook_sae_input): HookPoint(name='hook_sae_input')
  (hook_sae_acts_pre): HookPoint(name='hook_sae_acts_pre')
  (hook_sae_acts_post): HookPoint(name='hook_sae_acts_post')
  (hook_sae_output): HookPoint(name='hook_sae_output')
  (hook_sae_recons): HookPoint(name='hook_sae_recons')
  (hook_sae_error): HookPoint(name='hook_sae_error')
)

In [18]:
tokens = model.to_tokens("Hello, world!")
logits = model.run_with_saes(tokens, saes=[sae])

In [21]:
logits, cache = model.run_with_cache_with_saes(tokens, saes=[sae])

# Access SAE feature activations
sae_acts = cache["blocks.10.hook_resid_pre.hook_sae_acts_post"]
print(f"SAE activations shape: {sae_acts.shape}")

SAE activations shape: torch.Size([1, 5, 24576])


In [23]:
import json
from datasets import load_dataset

ds = load_dataset("stanfordnlp/sst2", split="train")

ds_with_id = ds.add_column("original_index", range(len(ds)))

first_split = ds_with_id.train_test_split(test_size=0.30, seed=42)

train_ds = first_split["train"]
temp_ds = first_split["test"]

second_split = temp_ds.train_test_split(test_size=0.5, seed=42)

val_ds = second_split["train"]
shap_ds = second_split["test"]

"""
train_indices = train_ds["original_index"]
val_indices = val_ds["original_index"]
shap_indices = shap_ds["original_index"]

indices_dict = {
    "train_indices": train_indices,
    "val_indices": val_indices,
    "shap_indices": shap_indices,
}


with open("three_way_split_indices.json", "w") as f:
    json.dump(indices_dict, f, indent=4)

print("Dataset three-way split complete!")
print(f"Train samples (70%): {len(train_indices)}")
print(f"Val samples   (15%): {len(val_indices)}")
print(f"Test samples  (15%): {len(shap_indices)}")
print("Indices saved successfully to 'three_way_split_indices.json'")
"""

README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

'\ntrain_indices = train_ds["original_index"]\nval_indices = val_ds["original_index"]\nshap_indices = shap_ds["original_index"]\n\nindices_dict = {\n    "train_indices": train_indices,\n    "val_indices": val_indices,\n    "shap_indices": shap_indices,\n}\n\n\nwith open("three_way_split_indices.json", "w") as f:\n    json.dump(indices_dict, f, indent=4)\n\nprint("Dataset three-way split complete!")\nprint(f"Train samples (70%): {len(train_indices)}")\nprint(f"Val samples   (15%): {len(val_indices)}")\nprint(f"Test samples  (15%): {len(shap_indices)}")\nprint("Indices saved successfully to \'three_way_split_indices.json\'")\n'

In [29]:
import numpy as np
import torch
from tqdm import tqdm


def extract_split_activations(model, sae, sentences, labels, batch_size=32):
    all_acts = []
    all_labels = []
    for i in tqdm(range(0, len(sentences), batch_size)):
        batch_sents = sentences[i : i + batch_size]
        batch_labels = labels[i : i + batch_size]
        tokens = model.to_tokens(batch_sents)  # [B, T]
        _, cache = model.run_with_cache_with_saes(tokens, saes=[sae])
        sae_acts = cache["blocks.10.hook_resid_pre.hook_sae_acts_post"]  # [B, T, F]
        # Last *real* token per sentence (not padding)
        pad_id = model.tokenizer.pad_token_id
        if pad_id is None:
            pad_id = model.tokenizer.eos_token_id
        # Count non-pad tokens per row; last real index = count - 1
        mask = tokens != pad_id
        last_idx = mask.sum(dim=1) - 1  # [B]
        batch_idx = torch.arange(sae_acts.shape[0], device=sae_acts.device)
        sentence_acts = sae_acts[batch_idx, last_idx, :]  # [B, F]
        all_acts.append(sentence_acts.cpu().numpy())
        all_labels.append(np.array(batch_labels))
    activations = np.concatenate(all_acts, axis=0)   # (n_sentences, 24576)
    labels_arr  = np.concatenate(all_labels, axis=0) # (n_sentences,)
    return activations, labels_arr

In [31]:
activations, labels_arr = extract_split_activations(model, sae, shap_ds["sentence"], shap_ds["label"])

100%|██████████| 316/316 [00:24<00:00, 12.68it/s]
